# Eminem Lyric Generator - RNN with PyTorch

This notebook implements a character-level RNN (using a GRU) to generate lyrics in the style of Eminem. 

### Steps:
1. **Preprocessing**: Load text and map characters to integers.
2. **Data Loading**: Create sequences for training.
3. **Model Architecture**: Define a multi-layer RNN (GRU).
4. **Training**: Implement the training loop.
5. **Generation**: Function to generate new text from a seed.

In [1]:
import torch
import torch.nn as nn
import numpy as np
import os

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 1. Preprocessing
We load the cleaned lyrics and create character mappings.

In [2]:
path_to_file = '../data/cleaned_eminem.txt'
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')

chars = sorted(list(set(text)))
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)

print(f"Total characters: {len(text)}")
print(f"Unique characters: {vocab_size}")

Total characters: 934026
Unique characters: 108


## 2. Model Architecture
We use an Embedding layer, a GRU (better than basic RNN for sequences), and a Linear output layer.

In [3]:
class LyricRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(LyricRNN, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, h):
        x = self.embed(x)
        out, h = self.gru(x, h)
        out = self.fc(out.reshape(out.size(0) * out.size(1), out.size(2)))
        return out, h

# Hyperparameters
embed_size = 256
hidden_size = 512
num_layers = 2
seq_length = 100
batch_size = 64
learning_rate = 0.001
num_epochs = 20

model = LyricRNN(vocab_size, embed_size, hidden_size, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

## 3. Training Loop
We slice the text into sequences and train the model to predict the next character.

In [4]:
# Prepare encoded data
encoded_text = np.array([char_to_int[ch] for ch in text])

def get_batches(data, batch_size, seq_length):
    n_batches = len(data) // (batch_size * seq_length)
    data = data[:n_batches * batch_size * seq_length]
    data = data.reshape((batch_size, -1))
    
    for n in range(0, data.shape[1], seq_length):
        x = data[:, n:n+seq_length]
        y = np.zeros_like(x)
        try:
            # We use int() cast to ensure Pylance/static checkers recognize it as an integer index
            y[:, :-1], y[:, -1] = x[:, 1:], data[:, int(n+seq_length)]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], data[:, 0]
        yield torch.tensor(x), torch.tensor(y)

# Training
model.train()
for epoch in range(num_epochs):
    h = None # Initial hidden state
    for i, (inputs, targets) in enumerate(get_batches(encoded_text, batch_size, seq_length)):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Forward pass
        outputs, h = model(inputs, h)
        # Detach hidden state to prevent backpropagating through the entire history
        h = h.detach()
        
        loss = criterion(outputs, targets.reshape(-1))
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (i+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}], Loss: {loss.item():.4f}')

Epoch [1/20], Step [100], Loss: 1.8560
Epoch [2/20], Step [100], Loss: 1.5851
Epoch [3/20], Step [100], Loss: 1.4659
Epoch [4/20], Step [100], Loss: 1.3860
Epoch [5/20], Step [100], Loss: 1.3230
Epoch [6/20], Step [100], Loss: 1.2667
Epoch [7/20], Step [100], Loss: 1.2151
Epoch [8/20], Step [100], Loss: 1.1689
Epoch [9/20], Step [100], Loss: 1.1286
Epoch [10/20], Step [100], Loss: 1.0901
Epoch [11/20], Step [100], Loss: 1.0576
Epoch [12/20], Step [100], Loss: 1.0295
Epoch [13/20], Step [100], Loss: 1.0081
Epoch [14/20], Step [100], Loss: 0.9708
Epoch [15/20], Step [100], Loss: 0.9476
Epoch [16/20], Step [100], Loss: 0.9199
Epoch [17/20], Step [100], Loss: 0.9106
Epoch [18/20], Step [100], Loss: 0.9002
Epoch [19/20], Step [100], Loss: 0.8829
Epoch [20/20], Step [100], Loss: 0.8573


## 4. Generation
Function to sample from the model.

In [12]:
def generate(model, start_str='Look', length=200, temperature=0.7):
    """
    Generates text using the trained model.
    - temperature: Higher values (e.g., 1.0) make output more random, lower (e.g., 0.2) more predictable.
    """
    model.eval()
    chars = [ch for ch in start_str]
    input_seq = torch.tensor([[char_to_int[ch] for ch in start_str]]).to(device)
    h = None
    
    for _ in range(length):
        output, h = model(input_seq, h)
        
        # Apply temperature scaling to the last character's predictions
        # output[-1] is the logit vector; dividing by temperature scales the distribution
        output_dist = (output[-1] / max(temperature, 1e-6)).exp()
        
        # Sample from the multinomial distribution
        top_ch_tensor = torch.multinomial(output_dist, 1)[0]
        top_ch = int(top_ch_tensor.item())
        
        # Append the character (ensuring the index is an explicit integer for Pylance)
        chars.append(int_to_char[top_ch])
        
        # Update the input sequence for the next step
        input_seq = torch.tensor([[top_ch]]).to(device)
        
    return ''.join(chars)

print(generate(model, start_str="Moms spaghetti", temperature=0.7))

Moms spaghetti (Yeah)
Now bring it backwards check it, relative mession
Yeah, haha
(What the fuck's what?)
You know, you acting like you don't like (hey), what'll be the girl (Yeah)
There's nothing left (Yeah)
